In [1]:
import torch
import gc

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

GPU memory cleared
GPU memory available: 15.83 GB
GPU memory allocated: 0.00 GB
GPU memory reserved: 0.00 GB


In [3]:
!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

import os
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re
import uuid
import shutil
from rank_bm25 import BM25Okapi


INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 66.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━

2025-11-16 09:48:00.782100: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763286480.990763      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763286481.055153      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
import os
import re
from typing import List, Optional
import torch
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document
from langchain import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from rank_bm25 import BM25Okapi


class LegalSearchAgent:
    # =======================================================================
    #                         INITIALIZATION
    # =======================================================================
    def __init__(self, pdf_folder: str, embeddings: HuggingFaceEmbeddings, db_path: str = "chroma_db", test_mode: bool = True):
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode
        self.embeddings = embeddings

        self.vectorstore: Optional[Chroma] = None
        self.bm25: Optional[BM25Okapi] = None
        self.bm25_docs: List[Document] = []
        self.bm25_corpus: List[List[str]] = []

        # Initialize LLM for answer generation
        self.llm = self._init_llm()

    def _init_llm(self) -> Optional[HuggingFacePipeline]:
        """Initialize LLM for answer generation"""
        try:
            
            model_name = "Qwen/Qwen2.5-3B-Instruct"
            print("🤖 Loading LLM for answer generation..."+ model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="auto",
                trust_remote_code=True
            )
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True,
                top_p=0.95
            )
            llm = HuggingFacePipeline(pipeline=pipe)
            print("✅ LLM loaded successfully")
            return llm
        except Exception as e:
            print(f"⚠️ Could not load Phi-2: {e}")
            print("Will use extractive answers only")
            return None

    # =======================================================================
    #                         CASE NUMBER DETECTION
    # =======================================================================
    def _detect_case_number(self, query: str) -> Optional[str]:
        pattern = r"(CPLA|C\.A\.|Cr\.A|C\.P\.|HCA|RFA)[\s\-]*\d+[\s/]*(?:of\s*)?\d{4}"
        match = re.search(pattern, query, re.IGNORECASE)
        if match:
            case = match.group(0)
            case = case.replace(" ", "_").replace("of_", "_").replace("/", "_")
            case = re.sub(r"__+", "_", case)
            return case
        return None

    # =======================================================================
    #                         HYBRID RETRIEVER
    # =======================================================================
    def _retrieve_documents(self, query: str, k: int = 10) -> List[Document]:
        if self.vectorstore is None or self.bm25 is None:
            print("⚠️ Vectorstore or BM25 not built yet.")
            return []

        # Semantic search
        semantic_results = self.vectorstore.similarity_search(query, k=k)

        # BM25 keyword search
        tokens = query.split()
        bm25_scores = self.bm25.get_scores(tokens)
        bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]
        bm25_results = [self.bm25_docs[i] for i in bm25_top_idx]

        # Merge results (remove duplicates by case_number)
        combined = {}
        for doc in semantic_results + bm25_results:
            key = doc.metadata.get("case_number", doc.metadata.get("source_file", id(doc)))
            if key not in combined:
                combined[key] = doc

        return list(combined.values())[:k]

    # =======================================================================
    #                         SEARCH + ANSWER
    # =======================================================================
    def search(self, query: str, k: int = 10) -> str:
        """Search for documents and generate a structured legal answer."""
        print(f"\n🔍 QUERY: {query}")

        # Detect case number in query
        case_num = self._detect_case_number(query)

        if case_num:
            print(f"📋 Detected case number: {case_num}")
            parts = case_num.split("_")
            if len(parts) >= 2:
                case_number_only = parts[-2]
                case_year = parts[-1]

                # Retrieve more results to allow filtering
                results = self._retrieve_documents(query, k=k*3)

                # Filter to exact case match (number + year)
                exact_match = [
                    r for r in results
                    if case_number_only in r.metadata.get('case_number', '') and case_year == r.metadata.get('case_year', '')
                ]

                if exact_match:
                    results = exact_match[:k]
                    print(f"✅ Found exact case match")
                else:
                    results = results[:k]
                    print(f"⚠️ No exact case found, returning top semantic matches")
            else:
                results = self._retrieve_documents(query, k=k)
        else:
            results = self._retrieve_documents(query, k=k)

        # Print only the filenames of retrieved PDFs
        print("\n📄 Retrieved PDFs:")
        for r in results:
            print(" -", r.metadata.get("source_file", "unknown"))

        # Generate answer using all retrieved documents
        answer_text = self.generate_answer(query, results)

        # Print clean structured answer
        print("\n📌 LLM Answer:\n", answer_text)

        return answer_text

    # =======================================================================
    #                         ANSWER GENERATION
    # =======================================================================
    def generate_answer(self, query: str, documents: List[Document], use_llm: bool = True) -> str:
        """Generate a structured answer using all retrieved documents."""
        if not documents:
            return "No relevant documents found."

        # Build context from all documents
        context = "\n\n---\n\n".join([
            f"From {d.metadata.get('source_file', 'unknown')}:\n{d.page_content}"
            for d in documents
        ])

        if self.llm and use_llm:
            prompt = f"""You are a Legal Case Retrieval and Question Answering Assistant. You answer strictly using the retrieved documents provided to you. Lawyers may describe a case, ask about similar cases, ask for details of a specific case, or request information such as parties, judges, years, procedural posture, or outcomes. Your job is to provide clear, reliable, structured answers using only the supplied retrieval context.

Your response must ALWAYS follow this exact structure:

1. Final Answer:
Provide a clear, concise legal answer to the user’s query in your own words. This must be the first section. Do not mention PDFs, retrieval steps, metadata, embeddings, or your reasoning. Just answer the question directly and professionally, based only on the retrieved documents.

2. Sources Used:
List ONLY the PDF filenames that directly contributed to your answer. One per line. No commentary, no extra text.

3. Verbatim Extracts:
Copy and paste the exact sentences or paragraphs from the PDFs that support your answer. These must be word-for-word quotes. Under each quote, clearly mention the PDF filename it comes from.

Rules:
- You must not hallucinate any information or case details that are not present in the retrieved documents.
- If the answer cannot be found in the retrieved documents, respond in the Final Answer section: "The retrieved documents do not contain the required information."
- Use only the content in the retrieved documents. Never add external legal knowledge.
"""

            # Append retrieved documents to prompt
            prompt += "\n\nRETRIEVED DOCUMENTS:\n" + context

            try:
                result = self.llm.invoke(prompt)
                if isinstance(result, list) and len(result) > 0:
                    answer_text = result[0].get("generated_text", str(result))
                elif isinstance(result, str):
                    answer_text = result
                else:
                    answer_text = str(result)
                return answer_text.strip()
            except Exception as e:
                print(f"⚠️ LLM error: {e}, falling back to extractive answer")
                return self._extractive_answer(documents)
        else:
            return self._extractive_answer(documents)

    def _extractive_answer(self, documents: List[Document]) -> str:
        """Fallback extractive answer if LLM fails."""
        answer = "Based on the retrieved documents:\n\n"
        for i, doc in enumerate(documents[:3], 1):
            answer += f"{i}. From {doc.metadata.get('source_file', 'unknown')}:\n"
            answer += f"   {doc.page_content[:300]}...\n\n"
        return answer


In [7]:
print("=" * 70)
print("COPYING DB TO WRITABLE LOCATION")
print("=" * 70)

# Copy from read-only input to writable workspace
src = "/kaggle/input/fyp-vector-store/chroma_db"
dst = "/kaggle/working/chroma_db"

if os.path.exists(dst):
    shutil.rmtree(dst)

print(f"\nCopying from: {src}")
print(f"Copying to: {dst}")
shutil.copytree(src, dst)
print("✅ Copied successfully")

# Now load from writable location
print("\n📚 Loading embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda'}
)

print("\n🔧 Initializing agent...")
agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",
    embeddings=embeddings,
    db_path="/kaggle/working/chroma_db",  # ← Use writable location
    test_mode=False
)

print("\n📦 Loading vector store...")
agent.vectorstore = Chroma(
    persist_directory="/kaggle/working/chroma_db",
    embedding_function=embeddings
)
chunk_count = agent.vectorstore._collection.count()
print(f"✅ Loaded {chunk_count} chunks")

print("\n🔨 Rebuilding BM25...")
vectorstore_data = agent.vectorstore.get()

docs = []
for i, content in enumerate(vectorstore_data['documents']):
    metadata = vectorstore_data['metadatas'][i]
    doc = Document(page_content=content, metadata=metadata)
    docs.append(doc)

agent.bm25_docs = docs
agent.bm25_corpus = [doc.page_content.split() for doc in docs]
agent.bm25 = BM25Okapi(agent.bm25_corpus)
print(f"✅ BM25 ready with {len(agent.bm25_docs)} documents")

print("\n" + "=" * 70)
print("✅ READY TO USE")
print("=" * 70)



COPYING DB TO WRITABLE LOCATION

Copying from: /kaggle/input/fyp-vector-store/chroma_db
Copying to: /kaggle/working/chroma_db
✅ Copied successfully

📚 Loading embeddings...


/tmp/ipykernel_48/3423715723.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0
/tmp/ipykernel_48/1782900801.py:55: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)
/tmp/ipykernel_48/3423715723.py:33: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  agent.vectorstore = Chroma(


✅ LLM loaded successfully

📦 Loading vector store...
✅ Loaded 21895 chunks

🔨 Rebuilding BM25...
✅ BM25 ready with 21895 documents

✅ READY TO USE


In [8]:
query = "What was CPLA 210 of 2024 about?"
answer = agent.search(query, k=5)
print(answer)


🔍 QUERY: What was CPLA 210 of 2024 about?
📋 Detected case number: CPLA_210_2024
✅ Found exact case match

📄 Retrieved PDFs:
 - C.P.L.A.210_2024.pdf

📌 LLM Answer:
 You are a Legal Case Retrieval and Question Answering Assistant. You answer strictly using the retrieved documents provided to you. Lawyers may describe a case, ask about similar cases, ask for details of a specific case, or request information such as parties, judges, years, procedural posture, or outcomes. Your job is to provide clear, reliable, structured answers using only the supplied retrieval context.

Your response must ALWAYS follow this exact structure:

1. Final Answer:
Provide a clear, concise legal answer to the user’s query in your own words. This must be the first section. Do not mention PDFs, retrieval steps, metadata, embeddings, or your reasoning. Just answer the question directly and professionally, based only on the retrieved documents.

2. Sources Used:
List ONLY the PDF filenames that directly contribu